# 2.1 章节介绍

## 概述

第 1 章中，我们照着示例跑通了 vecadd，但还没有具体解释这段代码为什么这样写。本章将从这个例子出发，先分清 Host、kernel、编译和调用之间的关系，再认识 `@tilelang.jit`、`@T.prim_func`、`T.Kernel` 和 `T.Tensor/T.Buffer` 这些常见写法；接着理解数据如何在 GM、UB 和寄存器之间流动，并独立完成 Add 的规格定义、实现和验证。最后通过 Mul、SAXPY 和 2-D Add 练习，把这套写法迁移到新的计算任务中。

## 学习目标

- 说清 Host 与 kernel 的边界。
- 用 CopyIn → Compute → CopyOut 组织一个简单 vector kernel。
- 独立完成 Add，并迁移到三档练习。

## 本章学习安排

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">小节</th>
<th style="text-align: left; vertical-align: top;">主题</th>
<th style="text-align: left; vertical-align: top;">建议用时</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="02.01_chapter_intro.ipynb">2.1</a></td>
<td style="text-align: left; vertical-align: top;">承上启下与目标</td>
<td style="text-align: left; vertical-align: top;">0.2 h</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="02.02_kernel_structure.ipynb">2.2</a></td>
<td style="text-align: left; vertical-align: top;">Host/kernel 边界、基本写法与调试原语</td>
<td style="text-align: left; vertical-align: top;">0.35 h</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="02.03_core_programming_model.ipynb">2.3</a></td>
<td style="text-align: left; vertical-align: top;">GM/UB/寄存器、搬运和向量计算</td>
<td style="text-align: left; vertical-align: top;">0.45 h</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="02.04_add_workflow.ipynb">2.4</a></td>
<td style="text-align: left; vertical-align: top;">Add 完整开发、compile 与 jit</td>
<td style="text-align: left; vertical-align: top;">0.6 h</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="02.05_chapter_practice.ipynb">2.5</a></td>
<td style="text-align: left; vertical-align: top;">Mul / SAXPY / 2-D Add 三档实践</td>
<td style="text-align: left; vertical-align: top;">0.4 h</td>
</tr>
</tbody>
</table>

<p><small>本章沿用<a href="../01_overview/01.04_first_operator.ipynb">1.4「环境准备与第一个算子」</a>中的环境准备方法：先在终端加载CANN环境，再运行示例脚本；也可以使用Notebook中的执行入口。线上环境不提供950设备，可先阅读正文；自行运行时，请使用匹配的950环境。</small></p>

## 本章学习主线

本章基于上一章已经实现并验证的 VecAdd，继续完成以下工作：

- **看清程序结构**：分清 Host 与 kernel 在哪里运行，理解 kernel 描述怎样经过编译，最后由 Host 发起调用。
- **认识基本写法**：结合 `@tilelang.jit`、`@T.prim_func`、`T.Kernel` 和 `T.Tensor/T.Buffer`，找到编译入口、计算描述、任务范围和数据接口。
- **串起数据通路**：跟踪一份数据从 GM 搬到 UB、进入寄存器完成计算，再写回 GM，理解 CopyIn → Compute → CopyOut。
- **独立完成 Add**：从规格和数据切分开始编写算子，并用 PyTorch 参考结果检查完整输出。
- **迁移同一套写法**：完成 Mul、SAXPY 和 2-D Add，为下一章学习 Tiling 和 Vector 算子优化打下基础。

## 小结

你知道本章要从“用过”走到“会写”，下一节直接解剖第 1 章的 vecadd。

## 课后练习

写下你认为属于 Host 的两项工作和属于 kernel 的两项工作，答案见 [02.01_questions.txt](answer/02.01_questions.txt)。